# BuildSafe-A
## Final Preprocessing Pipeline & Data-Leakage Prevention
**Project identifier:** Gima  · 

> **Main explanation:** The pipeline prevents target leakage and preprocessing leakage by ensuring that information from unseen test data does not influence training.

**Responsibilities covered:** separate `SEVERITY` as the target · separate input features · train-test split · fit preprocessing on training data only · apply the same transformations to test data · explain the final feature representation.

**Input:** `data/processed/buildsafe_features.csv` (Notebook 3).

```
raw CSV ─► Notebook 2: clean ─► Notebook 3: features + text scrub ─► NOTEBOOK 4:
      split by DATE ─► fit encoder / TF-IDF on TRAIN only ─► transform TEST with the same fitted objects ─► final matrices
```

> **About the outputs in this notebook.** Cells copied unchanged from the team's original notebooks keep the outputs from the original run on the real data.
> Cells that were newly written or reorganised for the viva have **no saved output** — run the notebook top-to-bottom (Kernel → Restart & Run All) before the evaluation so every number on screen is one you can explain.

## 1. Load the feature dataset

In [ ]:
import re
import numpy as np
import pandas as pd

DATA_PATH = "../data/processed/buildsafe_features.csv"

text_cols = {c: "string" for c in ["BIN", "BLOCK", "LOT", "RESPONDENT_CITY", "RESPONDENT_ZIP",
                                   "INFRACTION_CODE1", "SECTION_LAW_DESCRIPTION1", "VIOLATION_DESCRIPTION_CLEAN"]}
df = pd.read_csv(DATA_PATH, dtype=text_cols, parse_dates=["ISSUE_DATE_DT"])
df = df.sort_values("ISSUE_DATE_DT", kind="mergesort").reset_index(drop=True)

print("Shape:", df.shape)
print(list(df.columns))

Shape: (102467, 22)
['BIN', 'BORO', 'BLOCK', 'LOT', 'ISSUE_DATE', 'SEVERITY', 'VIOLATION_TYPE', 'RESPONDENT_CITY', 'RESPONDENT_ZIP', 'INFRACTION_CODE1', 'SECTION_LAW_DESCRIPTION1', 'ISSUE_DATE_DT', 'ISSUE_YEAR', 'ISSUE_MONTH', 'ISSUE_DAY', 'ISSUE_DAY_OF_WEEK', 'ISSUE_QUARTER', 'PREVIOUS_VIOLATION_COUNT', 'PREVIOUS_CLASS1_COUNT', 'PREVIOUS_CLASS2_COUNT', 'PREVIOUS_CLASS3_COUNT', 'VIOLATION_DESCRIPTION_CLEAN']


## 2. Target and input features

**What:** separate the target (`SEVERITY`) from the input features. **Why:** the answer must never be part of the inputs. **How:** `X = everything except SEVERITY`, `y = SEVERITY`.

| Group | Columns | Treatment |
|---|---|---|
| **Target** | `SEVERITY` | becomes `y` |
| **Numeric (9)** | `ISSUE_YEAR`, `ISSUE_MONTH`, `ISSUE_DAY`, `ISSUE_DAY_OF_WEEK`, `ISSUE_QUARTER`, `PREVIOUS_VIOLATION_COUNT`, `PREVIOUS_CLASS1/2/3_COUNT` | used as numbers |
| **Categorical (6)** | `BORO`, `VIOLATION_TYPE`, `RESPONDENT_CITY`, `RESPONDENT_ZIP`, `INFRACTION_CODE1`, `SECTION_LAW_DESCRIPTION1` | one-hot |
| **Text (1)** | `VIOLATION_DESCRIPTION_CLEAN` | TF-IDF (scrubbed text) |
| **Dropped** | `BIN`, `BLOCK`, `LOT`, `ISSUE_DATE`, `ISSUE_DATE_DT` | identifiers / replaced by engineered features |

In [ ]:
drop_features = [
    "BIN",
    "BLOCK",
    "LOT",
    "ISSUE_DATE",
    "ISSUE_DATE_DT"
]

In [ ]:
numeric_features = [
    "ISSUE_YEAR",
    "ISSUE_MONTH",
    "ISSUE_DAY",
    "ISSUE_DAY_OF_WEEK",
    "ISSUE_QUARTER",
    "PREVIOUS_VIOLATION_COUNT",
    "PREVIOUS_CLASS1_COUNT",
    "PREVIOUS_CLASS2_COUNT",
    "PREVIOUS_CLASS3_COUNT"
]

In [ ]:
categorical_features = [
    "BORO",
    "VIOLATION_TYPE",
    "RESPONDENT_CITY",
    "RESPONDENT_ZIP",
    "INFRACTION_CODE1",
    "SECTION_LAW_DESCRIPTION1"
]

In [ ]:
TARGET = "SEVERITY"
text_feature = "VIOLATION_DESCRIPTION_CLEAN"      # scrubbed text (raw description no longer exists in this file)

print("Dropped:    ", len(drop_features))
print("Numerical:  ", len(numeric_features))
print("Categorical:", len(categorical_features))
print("Text:       ", text_feature)

Dropped:     5
Numerical:   9
Categorical: 6
Text:        VIOLATION_DESCRIPTION_CLEAN


## 3. Train / test split — by **date**, not at random

**What:** train on the earlier ~80 % of the timeline and test on the latest ~20 %.
**Why (three reasons):**
1. **It mimics real use.** The model will always predict *future* violations from *past* ones. A random split would let it learn from violations issued *after* some of its test rows.
2. **Building history.** The history features are built from earlier records of the same building. A random split would put a building's Monday and Tuesday violations in different sets and blur the boundary.
3. **Same-day batches.** Many violations of one building are issued on the same day (61 for one building). Random splitting would place near-identical siblings in both train and test, inflating scores.

**How:** sort by `ISSUE_DATE_DT`, find the 80 % position, then **use that date as the boundary** so that no date appears on both sides.

In [ ]:
split_index = int(len(df) * 0.80)

train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

print("Training shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining period:")
print(
    train_df["ISSUE_DATE_DT"].min(),
    "→",
    train_df["ISSUE_DATE_DT"].max()
)

print("\nTesting period:")
print(
    test_df["ISSUE_DATE_DT"].min(),
    "→",
    test_df["ISSUE_DATE_DT"].max()
)

Training shape: (81973, 22)
Testing shape: (20494, 22)

Training period:
2025-01-01 00:00:00 → 2026-05-13 00:00:00

Testing period:
2026-05-13 00:00:00 → 2026-09-10 00:00:00


The first attempt cuts at row 81,973 — in the *middle* of a date, so that day's violations would sit in both sets (train ends 2026-05-13, test begins 2026-05-13). Fix: cut on the **date** itself.

In [ ]:
# ============================================================
# FINAL TEMPORAL SPLIT — DATE BOUNDARY
# ============================================================

# Use the date at the current 80% boundary
split_date = df.iloc[split_index]["ISSUE_DATE_DT"].normalize()

# Training = dates BEFORE split date
train_mask = df["ISSUE_DATE_DT"] < split_date

# Testing = dates ON/AFTER split date
test_mask = df["ISSUE_DATE_DT"] >= split_date

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()

print("Split date:", split_date)

print("\nTraining shape:", train_df.shape)
print("Testing shape:", test_df.shape)

print("\nTraining period:")
print(train_df["ISSUE_DATE_DT"].min(), "→", train_df["ISSUE_DATE_DT"].max())

print("\nTesting period:")
print(test_df["ISSUE_DATE_DT"].min(), "→", test_df["ISSUE_DATE_DT"].max())

print("\nSame dates in both sets:",
      len(set(train_df["ISSUE_DATE_DT"]) &
          set(test_df["ISSUE_DATE_DT"])))

Split date: 2026-05-13 00:00:00

Training shape: (81903, 22)
Testing shape: (20564, 22)

Training period:
2025-01-01 00:00:00 → 2026-05-12 00:00:00

Testing period:
2026-05-13 00:00:00 → 2026-09-10 00:00:00

Same dates in both sets: 0


In [ ]:
TARGET = "SEVERITY"

X_train = train_df.drop(columns=[TARGET])
y_train = train_df[TARGET]

X_test = test_df.drop(columns=[TARGET])
y_test = test_df[TARGET]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test: ", X_test.shape)
print("y_test: ", y_test.shape)

X_train: (81903, 21)
y_train: (81903,)
X_test:  (20564, 21)
y_test:  (20564,)


In [ ]:
print("TRAINING TARGET")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTEST TARGET")
print(y_test.value_counts())
print(y_test.value_counts(normalize=True).mul(100).round(2))

TRAINING TARGET
SEVERITY
CLASS - 2    46320
CLASS - 1    32213
CLASS - 3     3370
Name: count, dtype: int64
SEVERITY
CLASS - 2    56.55
CLASS - 1    39.33
CLASS - 3     4.11
Name: proportion, dtype: float64

TEST TARGET
SEVERITY
CLASS - 2    11287
CLASS - 1     8460
CLASS - 3      817
Name: count, dtype: int64
SEVERITY
CLASS - 2    54.89
CLASS - 1    41.14
CLASS - 3     3.97
Name: proportion, dtype: float64


**Observation.** Train = 2025-01-01 → 2026-05-12 (81,903 rows), test = 2026-05-13 → 2026-09-10 (20,564 rows), **0 shared dates**. Class proportions are similar in both sets (Class 3 ≈ 4 % each), so the rare class is represented in the test set too.

## 4. Fit preprocessing on training data **only**

**What:** learn the vocabulary (TF-IDF) and the category lists (one-hot) from the training rows only, then use those *fitted* objects to transform the test rows.
**Why:** if TF-IDF or the encoder were fitted on all data, test-set information (its words, its rare categories, its document frequencies) would influence the features the model trains on → **preprocessing leakage**, and test scores would look better than reality.
**How:** `fit_transform(train)` for training data, `transform(test)` for test data — `fit` is **never** called on test data.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=True
)

print("Categorical encoder created.")

Categorical encoder created.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

print("TF-IDF vectorizer created.")

TF-IDF vectorizer created.


### 4.1 Text: TF-IDF

In [ ]:
# FIT TF-IDF USING TRAINING DATA ONLY
X_train_text = X_train[text_feature].fillna("")
X_test_text  = X_test[text_feature].fillna("")

tfidf_train = tfidf.fit_transform(X_train_text)      # learns vocabulary + IDF from TRAIN
tfidf_test  = tfidf.transform(X_test_text)           # re-uses TRAIN vocabulary + IDF

print("TF-IDF training matrix:", tfidf_train.shape)
print("TF-IDF test matrix:    ", tfidf_test.shape)
print("Vocabulary learned from training data:", len(tfidf.get_feature_names_out()), "terms")

TF-IDF training matrix: (81903, 5000)
TF-IDF test matrix:     (20564, 5000)
Vocabulary learned from training data: 5000 terms


### 4.2 Categorical: one-hot

In [ ]:
# ============================================================
# FIT CATEGORICAL ENCODER ON TRAINING DATA
# ============================================================

X_train_cat = X_train[categorical_features].copy()
X_test_cat = X_test[categorical_features].copy()

# Convert to string so all categorical columns have a consistent type
X_train_cat = X_train_cat.astype(str)
X_test_cat = X_test_cat.astype(str)

# Fit ONLY on training data
cat_train = categorical_encoder.fit_transform(X_train_cat)

# Transform test using the same fitted encoder
cat_test = categorical_encoder.transform(X_test_cat)

print("Categorical training matrix:", cat_train.shape)
print("Categorical testing matrix: ", cat_test.shape)
print("Number of encoded categories:", cat_train.shape[1])

Categorical training matrix: (81903, 3011)
Categorical testing matrix:  (20564, 3011)
Number of encoded categories: 3011


**Observation.** 3,011 one-hot columns were learned from the training rows. Categories that appear only in the test period get all zeros (`handle_unknown="ignore"`) — evidence in §5.2.

### 4.3 Numeric

In [ ]:
# ============================================================
# NUMERICAL FEATURES
# ============================================================

X_train_num = X_train[numeric_features].astype(float).values
X_test_num = X_test[numeric_features].astype(float).values

print("Numerical training matrix:", X_train_num.shape)
print("Numerical testing matrix: ", X_test_num.shape)

Numerical training matrix: (81903, 9)
Numerical testing matrix:  (20564, 9)


### 4.4 Combine into the final feature matrix

In [ ]:
from scipy.sparse import hstack, csr_matrix

X_train_final = hstack([csr_matrix(X_train_num), cat_train, tfidf_train], format="csr")
X_test_final  = hstack([csr_matrix(X_test_num),  cat_test,  tfidf_test],  format="csr")

print("Final X_train:", X_train_final.shape)
print("Final X_test: ", X_test_final.shape)
print("y_train:", y_train.shape, "| y_test:", y_test.shape)

blocks = pd.Series({"numeric": X_train_num.shape[1],
                    "categorical (one-hot)": cat_train.shape[1],
                    "text (TF-IDF)": tfidf_train.shape[1]})
print("\nFeature blocks:\n", blocks, "\nTotal:", blocks.sum())

Final X_train: (81903, 8020)
Final X_test:  (20564, 8020)
y_train: (81903,) | y_test: (20564,)

Feature blocks:
 numeric                     9
categorical (one-hot)    3011
text (TF-IDF)            5000
dtype: int64 
Total: 8020


**Final feature representation (original run: 9 + 3,011 + 5,000 = 8,020 columns).**
Every row is a sparse vector: 9 numeric values, then the one-hot flags, then TF-IDF weights. `X_train_final` and `X_test_final` have identical column layouts because they were produced by the same fitted objects.

## 5. Data-leakage audit

Leakage means the model gets information it will **not** have when predicting real, new violations. Six checks:

| # | Risk | What we did | Evidence |
|---|---|---|---|
| 1 | **Target leakage** — the answer is in the inputs | `SEVERITY` is removed from `X` | cell 5.1 |
| 2 | **Post-issue information** — hearing, status, payment | `SERVED_DATE`, `HEARING_*`, `ECB_VIOLATION_STATUS`, `CERTIFICATION_STATUS`, `PENALITY_IMPOSED`, `AMOUNT_PAID`, `BALANCE_DUE` dropped (Notebook 2) | Notebook 2 §5 |
| 3 | **Unknown-timing field** — `AGGRAVATED_LEVEL` | excluded (Level 2 = 82 % Class 1; timing cannot be proven) | Notebook 2 §8 |
| 4 | **Label written inside the text** | 4,273 "CLASS 1/2/3" mentions scrubbed | cell 5.3 |
| 5 | **Preprocessing leakage** — fitting on test data | encoder and TF-IDF fitted on train only | cell 5.2 |
| 6 | **Time leakage** — future data in training | split by date; history uses strictly earlier dates | §3, Notebook 3 §2 |

### 5.1 Target is not in the inputs

In [ ]:
print("Target in X_train columns:", TARGET in X_train.columns)
print("Target in X_test columns: ", TARGET in X_test.columns)
assert TARGET not in X_train.columns and TARGET not in X_test.columns
print("Number of final features:", X_train_final.shape[1])

Target in X_train columns: False
Target in X_test columns:  False
Number of final features: 8020


### 5.2 Fitted on training data only — evidence
The fitted objects contain **only** what the training rows contained. Test rows contain categories the encoder never saw:

In [ ]:
unseen = {c: int((~X_test_cat[c].isin(set(X_train_cat[c]))).sum()) for c in categorical_features}
print("Test rows whose category was NOT present in the training data:")
print(pd.Series(unseen).rename("unseen_in_train"))

# terms (unigrams + bigrams) that occur ONLY in the test text must not be in the vocabulary
analyzer   = tfidf.build_analyzer()
train_terms = {t for doc in X_train_text for t in analyzer(doc)}
test_terms  = {t for doc in X_test_text  for t in analyzer(doc)}
test_only   = test_terms - train_terms

print("\nTerms that exist only in the test text:", len(test_only))
print("…of which are in the TF-IDF vocabulary (must be 0):", len(test_only & set(tfidf.vocabulary_)))

Test rows whose category was NOT present in the training data:
BORO                          0
VIOLATION_TYPE                0
RESPONDENT_CITY             272
RESPONDENT_ZIP              235
INFRACTION_CODE1             22
SECTION_LAW_DESCRIPTION1     20
Name: unseen_in_train, dtype: int64

Terms that exist only in the test text: 66883
…of which are in the TF-IDF vocabulary (must be 0): 0


Those unseen categories are simply encoded as all-zeros — exactly what will happen with brand-new categories after deployment.

### 5.3 Severity words removed from the text

In [ ]:
pat_severity = r'(?i)\bclass\s*[-:#]?\s*(?:[123]|one|two|three|i{1,3})\b|\bclass[123]\b'

print("Severity mentions left in TRAIN text:", int(X_train_text.str.contains(pat_severity, regex=True).sum()))
print("Severity mentions left in TEST text: ", int(X_test_text.str.contains(pat_severity, regex=True).sum()))

Severity mentions left in TRAIN text: 0
Severity mentions left in TEST text:  0


Both must be **0**.

### 5.4 Checking for identical rows across the split
The team's original check compared complete feature records and found **0 overlap**. Be careful how you explain this: that check included the date, and train and test have *different dates by construction*, so it could not have found anything. A more informative version ignores the date and the history counts:

In [ ]:
content_cols = categorical_features + [text_feature]      # what the model actually "reads", without dates / history

train_keys = set(train_df[content_cols].astype(str).agg("|".join, axis=1))
test_keys  = set(test_df[content_cols].astype(str).agg("|".join, axis=1))

overlap = train_keys & test_keys
print("Distinct test records (ignoring date/history):", len(test_keys))
print("…identical to a training record:", len(overlap), f"({len(overlap)/len(test_keys):.1%})")

Distinct test records (ignoring date/history): 20378
…identical to a training record: 60 (0.3%)


A high overlap here is **not** leakage (the test row is a genuinely later violation) — but it tells us the description and codes repeat a lot, so models may partly memorise boilerplate. It is worth stating in the report.

### 5.5 ⚠️ The most important finding: the infraction code almost *is* the label
In the original modelling run a plain linear SVM reached **99.5 % accuracy** — suspiciously high for a problem where logistic regression scored 88 %. The text scrub does not explain it (the team's benchmark changed accuracy by only +0.09 points). The likelier cause is below: each ECB infraction code appears to be tied to a violation class (confirm against the DOB infraction-code table before stating this in the report), so `INFRACTION_CODE1` (and the legal-section text that goes with it) is close to a **lookup key for the target**.

In [ ]:
# ============================================================
# INFRACTION CODE → SEVERITY PURITY
# ============================================================

infraction_counts = pd.crosstab(
    df["INFRACTION_CODE1"],
    df["SEVERITY"]
)

infraction_purity = (
    infraction_counts.max(axis=1) /
    infraction_counts.sum(axis=1)
)

print("Total unique infraction codes:",
      len(infraction_counts))

print("\nCodes appearing with only ONE severity class:",
      (infraction_purity == 1.0).sum())

print("\nPercentage of codes with 100% purity:",
      round((infraction_purity == 1.0).mean() * 100, 2), "%")

print("\nLowest purity codes:")
print(infraction_purity.sort_values().head(20))

Total unique infraction codes: 326

Codes appearing with only ONE severity class: 250

Percentage of codes with 100% purity: 76.69 %

Lowest purity codes:
INFRACTION_CODE1
379    0.500000
2K4    0.611111
236    0.666667
2L3    0.707317
276    0.750000
2P2    0.750000
222    0.777778
382    0.789474
274    0.800000
2D3    0.818182
2D9    0.833333
2L2    0.857143
233    0.875000
244    0.888889
2N8    0.897959
2Q1    0.900000
2N7    0.909091
2G3    0.916667
2N3    0.916667
212    0.921933
dtype: float64


**Reading this:** 250 of 326 infraction codes (76.7 %) appear with only **one** severity class in the data. (This is descriptive analysis on all rows; it did not feed any model — the decision to keep the column came from the domain reasoning below.)

**Is it leakage?** Not in the *time* sense — the code is assigned when the violation is issued, so it is available at prediction time. But it makes the task partly trivial: a model can score very high by re-discovering a rule that the city already publishes. This is a **design decision to discuss with the instructor**, not something to hide:

| Option | Effect |
|---|---|
| **Keep** `INFRACTION_CODE1` (current pipeline) | high accuracy; the system is realistic *if* the code really is known at issue time; but scores overstate "learning" |
| **Remove** it and the linked legal-section text | harder, more honest problem; shows what the *other* signals (type, history, text, location) can do |
| **Report both** | best for the technical report: "with code" vs "without code" |

The optional experiment in §7 measures this directly.

## 6. Known gaps to state honestly (and fix before Evaluation 2)

1. **Numeric scaling.** The nine numeric features are used unscaled. That is fine for tree models and Naive Bayes, but the earlier logistic-regression run hit the iteration limit ("did not converge") — a sign that scaling is needed for linear models. Fix: a `MaxAbsScaler`/`StandardScaler` **fitted on the training numeric block only** (same leakage rule).
2. **Saved model files are out of date.** The pickles in `models/` (`final_svm.pkl`, `tfidf_vectorizer.pkl`, …) were produced from the *unscrubbed* description. They must be regenerated from this pipeline before the model comparison (Evaluation 2) and the app.
3. **History window.** History is truncated at Jan 2025; early rows therefore look "cleaner" than they are.
4. **`INFRACTION_CODE1` dependence** (§5.5).

## 7. Optional experiment — how much does the infraction code matter?

> ⚠️ This previews **Evaluation 2** (modelling). It is here only to measure the leakage-like effect above. Skip it if you are short of time; it may take a few minutes.

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

def fit_and_score(cat_cols, label):
    enc = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    c_tr = enc.fit_transform(X_train[cat_cols].astype(str))          # fit on TRAIN only
    c_te = enc.transform(X_test[cat_cols].astype(str))
    Xtr = hstack([csr_matrix(X_train_num), c_tr, tfidf_train], format="csr")
    Xte = hstack([csr_matrix(X_test_num),  c_te, tfidf_test],  format="csr")
    clf = LinearSVC(C=1.0, class_weight="balanced", max_iter=3000, random_state=42).fit(Xtr, y_train)
    pred = clf.predict(Xte)
    return {"setup": label, "accuracy": round(accuracy_score(y_test, pred), 4),
            "macro_F1": round(f1_score(y_test, pred, average="macro"), 4)}

with_code    = fit_and_score(categorical_features, "with INFRACTION_CODE1 + SECTION_LAW_DESCRIPTION1")
without_code = fit_and_score([c for c in categorical_features
                              if c not in ("INFRACTION_CODE1", "SECTION_LAW_DESCRIPTION1")],
                             "without them")
pd.DataFrame([with_code, without_code])

,setup,accuracy,macro_F1
0,with INFRACTION_CODE1 + SECTION_LAW_DESCRIPTION1,0.9927,0.9942
1,without them,0.8008,0.7437


Interpretation guide: a **large drop** means the codes carry most of the predictive power (label proxy); a **small drop** means the other features are genuinely informative. Either result is reportable. Note the TF-IDF text can still contain code-like wording (legal-section phrases), so "without" is an *upper bound* on how hard the honest problem is.

## 8. Viva preparation — Member 4

**Q1. What is the target and what are the features?** Target = `SEVERITY` (3 classes). Features = 9 numeric (date parts + building history), 6 categorical (one-hot), 1 text (TF-IDF of the scrubbed description). Identifiers and post-issue columns are excluded.

**Q2. Why split by date instead of randomly?** New violations always come *after* the ones we learn from. A random split would use the future to predict the past and put same-day sibling violations on both sides, inflating results. The date boundary guarantees no shared dates (0 verified).

**Q3. What does "fit only on training data" mean?** The TF-IDF vocabulary/IDF weights and the one-hot category lists are learned from the training rows alone. Test rows are only *transformed* with those fixed objects. Otherwise test information would shape the training features.

**Q4. What happens to a category or word that appears only in the test data?** One-hot gives all zeros (`handle_unknown="ignore"`); TF-IDF ignores words outside the training vocabulary. Shown in §5.2.

**Q5. What kinds of leakage did you look for?** Target leakage, post-issue columns, unknown-timing fields (`AGGRAVATED_LEVEL`), labels inside free text, preprocessing leakage, and time leakage. Each has a specific action (table in §5).

**Q6. How did you find the leakage in the text?** By searching `VIOLATION_DESCRIPTION` for "class": 7,378 rows; 4,273 explicitly state a severity class. We scrubbed only those patterns, keeping `CLASS A` and `CLASSROOM`.

**Q7. Your SVM got 99.5 % — do you trust it?** Not without qualification. It is far above logistic regression (88 %); the text scrub explains only a tiny part; 76.7 % of infraction codes map to a single severity class, so the code behaves like a label lookup. We flag it, test it with/without the code (§7) and will report both.

**Q8. What is the final feature representation?** A sparse matrix: 9 numeric + 3,011 one-hot + 5,000 TF-IDF = 8,020 columns in the original run, identical layout for train and test.

**Q9. What would you improve?** Scale the numeric block (fitted on train), regenerate the saved model files from this pipeline, and report results with and without the infraction code.